# **Automated Process for Training, Validation, and Testing of 2D U-Net Neural Network**

**Content under Creative Commons Attribution license CC-BY-NC-SA 4.0**  
**Code under GNU-GPL v3 License**  
**© 2024 Francesco Chiumento**

---

This notebook is part of the paper: 
> Chiumento F. et al. *Reducing Annotation Burden for Femoral Cartilage Segmentation in Knee MRI via Cross-Sequence Transfer Learning*  
> Preprint:  
> Submitted for peer review

---

# Rename image files

The following cell ensures that images and masks have the same name. After preprocessing with pykneer, "01_" and "_prep" are added to the images, which need to be removed.

**Set the path of the folder where the patients are located "images_dir".**

In [ ]:
import os

def rename_files(directory):
    for filename in os.listdir(directory):
        # Check if the file ends with the desired extension (e.g., '.mha')
        if filename.endswith('.mha'):
            # Extract the new name based on the removal of the '01_' prefix and the '_prep' suffix
            new_name = filename
            if new_name.startswith("01_"):
                new_name = new_name[3:]
            if new_name.endswith("_prep.mha"):
                new_name = new_name[:-9] + '.mha'

            # Construct the full path for the old and new file names
            old_file = os.path.join(directory, filename)
            new_file = os.path.join(directory, new_name)

            # Rename the file
            os.rename(old_file, new_file)
            print(f"Renamed '{filename}' to '{new_name}'")

# Directory containing the files to be renamed
images_dir =  r'YOUR_PATH_HERE/patients/images'
rename_files(images_dir)

# Splitting datasets

The following cell allows for the automatic splitting of datasets into 70% training, 20% validation, and 10% testing. An Excel file with all combinations will be created. From this file, the network can extract patients for each training session.

**Set the path of the folder where the patients are located "patients_dir" and where you want to save the Excel file "output_file".**

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def create_patient_split_excel(patients_dir, output_file, test_size=0.1, val_size=0.2):
    # Check if the file already exists and delete it
    if os.path.exists(output_file):
        os.remove(output_file)
        print(f"Existing file {output_file} deleted.")

    patients = sorted(os.listdir(patients_dir))
    num_patients = len(patients)
    p_out = int(num_patients * test_size)  # Number of patients in the test set per iteration
    num_iterations = num_patients // p_out  # Number of iterations

    data = {
        'Iteration': [],
        'Train_Patients': [],
        'Val_Patients': [],
        'Test_Patients': []
    }

    # Shuffle patients to ensure randomization
    np.random.shuffle(patients)

    for i in range(num_iterations):
        test_patients = patients[i * p_out: (i + 1) * p_out]
        remaining_patients = patients[:i * p_out] + patients[(i + 1) * p_out:]

        train_patients, val_patients = train_test_split(remaining_patients, test_size=val_size/(1 - test_size))

        data['Iteration'].append(i + 1)
        data['Train_Patients'].append(train_patients)
        data['Val_Patients'].append(val_patients)
        data['Test_Patients'].append(test_patients)

    df = pd.DataFrame(data)
    df.to_excel(output_file, index=False)

# Usage
patients_dir = r'YOUR_PATH_HERE/patients/images'  # Replace with the path to the directory containing all the patients
output_file = r'YOUR_PATH_HERE/patients_split.xlsx'
create_patient_split_excel(patients_dir, output_file)

# Below, set the main working folder that contains codes and folders.

In [ ]:
%cd /YOUR_PATH_HERE

# Correct setting of paths and running the code.

**Below, set the correct path of the Excel file with patient combinations in "*df*", and set the main working folder "*base_dir*" where the codes and folders are located.**

In [ ]:
import pandas as pd
import os
import shutil
import subprocess
import torch
import pexpect
import sys
import gc
from slice_extraction_nifti import run_slice_extraction
from train import main as train_main
from unet_testing import test_main

# Function to prepare directories and copy patient files
def prepare_directories(patients, iteration_dir, data_dir, set_type='train'):
    image_dir = os.path.join(iteration_dir, f'{set_type}_images')
    mask_dir = os.path.join(iteration_dir, f'{set_type}_masks')

    if not os.path.exists(image_dir):
        os.makedirs(image_dir)
    if not os.path.exists(mask_dir):
        os.makedirs(mask_dir)

    for patient in patients:
        src_image = os.path.join(data_dir, 'images', patient)
        src_mask_nii = os.path.join(data_dir, 'masks', patient.replace('.mha', '.nii.gz'))
        src_mask_mha = os.path.join(data_dir, 'masks', patient)

        print(f"Processing patient: {patient}")

        if os.path.exists(src_image):
            shutil.copy(src_image, os.path.join(image_dir, patient))
            print(f"Copied image file for patient {patient}")
        else:
            print(f"Image file for patient {patient} not found in {src_image}. Skipping.")
            continue

        if os.path.exists(src_mask_nii):
            shutil.copy(src_mask_nii, os.path.join(mask_dir, patient.replace('.mha', '.nii.gz')))
            print(f"Copied NII mask file for patient {patient}")
        elif os.path.exists(src_mask_mha):
            shutil.copy(src_mask_mha, os.path.join(mask_dir, patient))
            print(f"Copied MHA mask file for patient {patient}")
        else:
            print(f"Mask file for patient {patient} not found in {src_mask_nii} or {src_mask_mha}. Skipping.")

    return image_dir, mask_dir

# Function to clean folders
def clean_folder(folder_path):
    if os.path.exists(folder_path):
        for file in sorted(os.listdir(folder_path), key=lambda x: x.lower()):
            file_path = os.path.join(folder_path, file)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')

def run_training_script():
    child = pexpect.spawn('python train.py', encoding='utf-8', timeout=None)  # Set timeout to None for no timeout
    child.logfile = sys.stdout  # This will print the output in real time
    child.expect(pexpect.EOF)
    child.close()

# Main function
def main():
    # Load the Excel file
    df = pd.read_excel('YOUR_PATH_HERE/patients_split.xlsx')

    base_dir = 'YOUR_PATH_HERE'
    data_dir = os.path.join(base_dir, 'patients')
    division_dir = os.path.join(base_dir, 'dataset_splits')
    checkpoints_dir = os.path.join(base_dir, 'checkpoints_saved')
    segmentations_dir = os.path.join(base_dir, 'segmentations')
    checkpoint_path = os.path.join(base_dir, 'my_checkpoint.pth.tar')
    to_segment_dir = os.path.join(base_dir, 'to_segment')
    ground_truth_dir = os.path.join(base_dir, 'ground_truth')
    prediction_dir = os.path.join(base_dir, 'prediction')
    postprocessed_dir = os.path.join(segmentations_dir, 'postprocessed')
    dicom_headers_dir = os.path.join(base_dir, 'dicom_header')

    if not os.path.exists(checkpoints_dir):
        os.makedirs(checkpoints_dir)
    if not os.path.exists(postprocessed_dir):
        os.makedirs(postprocessed_dir)

    # Ensure that the 'data' folder exists in the main directory
    output_data_dir = os.path.join(base_dir, 'data')
    os.makedirs(output_data_dir, exist_ok=True)
    os.makedirs(os.path.join(output_data_dir, 'train_images'), exist_ok=True)
    os.makedirs(os.path.join(output_data_dir, 'train_masks'), exist_ok=True)
    os.makedirs(os.path.join(output_data_dir, 'val_images'), exist_ok=True)
    os.makedirs(os.path.join(output_data_dir, 'val_masks'), exist_ok=True)

    for index, row in df.iterrows():
        iteration = row['Iteration']
        train_patients = eval(row['Train_Patients'])
        val_patients = eval(row['Val_Patients'])
        test_patients = eval(row['Test_Patients'])

        # Add .mha suffix to patient names only if it is not already present
        train_patients = [f"{patient}.mha" if not patient.endswith('.mha') else patient for patient in train_patients]
        val_patients = [f"{patient}.mha" if not patient.endswith('.mha') else patient for patient in val_patients]
        test_patients = [f"{patient}.mha" if not patient.endswith('.mha') else patient for patient in test_patients]

        # Debug print: verify patients read from Excel file
        print(f"Iteration: {iteration}")
        print(f"Train Patients: {train_patients}")
        print(f"Val Patients: {val_patients}")
        print(f"Test Patients: {test_patients}")

        # Directory for the current iteration
        iteration_dir = os.path.join(division_dir, f'iteration_{iteration}')
        if os.path.exists(iteration_dir):
            shutil.rmtree(iteration_dir)
        os.makedirs(iteration_dir)

        try:
            # Prepare directories and copy training, validation, and testing files
            train_image_dir, train_mask_dir = prepare_directories(train_patients, iteration_dir, data_dir, set_type='train')
            val_image_dir, val_mask_dir = prepare_directories(val_patients, iteration_dir, data_dir, set_type='val')
            test_image_dir, test_mask_dir = prepare_directories(test_patients, iteration_dir, data_dir, set_type='test')

            # Output directories for slices
            output_train_image_dir = os.path.join(output_data_dir, 'train_images')
            output_train_mask_dir = os.path.join(output_data_dir, 'train_masks')
            output_val_image_dir = os.path.join(output_data_dir, 'val_images')
            output_val_mask_dir = os.path.join(output_data_dir, 'val_masks')

            # Clean slice folders
            clean_folder(output_train_image_dir)
            clean_folder(output_train_mask_dir)
            clean_folder(output_val_image_dir)
            clean_folder(output_val_mask_dir)

            # Run slice extraction
            run_slice_extraction(train_image_dir, train_mask_dir, val_image_dir, val_mask_dir, output_train_image_dir, output_train_mask_dir, output_val_image_dir, output_val_mask_dir)

            # Run training
            run_training_script()

            # Save checkpoint
            checkpoint_src = os.path.join(base_dir, 'my_checkpoint.pth.tar')
            checkpoint_dst = os.path.join(checkpoints_dir, f'iteration_{iteration}_checkpoint.pth.tar')
            if os.path.exists(checkpoint_src):
                shutil.copy(checkpoint_src, checkpoint_dst)
            else:
                print(f"Checkpoint not found for iteration {iteration}")

            # Run testing
            test_images_dir = os.path.join(iteration_dir, 'test_images')
            test_masks_dir = os.path.join(iteration_dir, 'test_masks')
            test_main(test_images_dir, test_masks_dir, to_segment_dir, ground_truth_dir, checkpoint_path, prediction_dir, segmentations_dir, postprocessed_dir, dicom_headers_dir)

            # Remove checkpoint after testing
            if os.path.exists(checkpoint_src):
                os.remove(checkpoint_src)
                print(f"Checkpoint for iteration {iteration} has been removed after testing.")

            torch.cuda.empty_cache()
            gc.collect()

        finally:
            # Delete the current iteration directory to free up disk space
            if os.path.exists(iteration_dir):
                shutil.rmtree(iteration_dir)

if __name__ == "__main__":
    main()

In [ ]:
%load_ext watermark

# Display system information and package versions
%watermark -v -m -p numpy,pandas,torch,torchvision,scipy,matplotlib,skimage,albumentations,PIL,SimpleITK,nibabel,ipywidgets,psutil,GPUtil

# Additional script to gather detailed GPU information
import GPUtil
import platform

def get_size(bytes, suffix="B"):
    """
    Scale bytes to its proper format
    e.g:
        1253656 => '1.20MB'
        1253656678 => '1.17GB'
    """
    factor = 1024
    for unit in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f}{unit}{suffix}"
        bytes /= factor

def get_gpu_info():
    try:
        gpus = GPUtil.getGPUs()
        gpu_info = []
        for gpu in gpus:
            gpu_info.append({
                'GPU id': gpu.id,
                'GPU name': gpu.name,
                'GPU load': f"{gpu.load*100:.2f}%",
                'GPU free memory': f"{gpu.memoryFree}MB",
                'GPU used memory': f"{gpu.memoryUsed}MB",
                'GPU total memory': f"{gpu.memoryTotal}MB",
                'GPU temperature': f"{gpu.temperature}°C",
                'GPU uuid': gpu.uuid
            })
        return gpu_info
    except Exception as e:
        return str(e)

def print_gpu_info():
    gpu_info = get_gpu_info()
    if isinstance(gpu_info, list):
        print("GPU Information:")
        for gpu in gpu_info:
            print(f"GPU id: {gpu['GPU id']}")
            for key, value in gpu.items():
                if key != 'GPU id':
                    print(f"  {key:<17}: {value}")
    else:
        print("Error:", gpu_info)

print_gpu_info()

---
## Dependencies

In [ ]:
%load_ext watermark

%watermark
%watermark --iversions

---
<a name="attribution"></a>

Notebook created using the [template](https://github.com/ORMIRcommunity/templates/blob/main/ORMIR_nb_template.ipynb) of the [ORMIR community](https://ormircommunity.github.io/) (version 1.0, 2023)